In [1]:
# ---------------------------------------------------------
# MCSL-069 Assignment
# Q1: N-Queen Problem without using Recursion
# ---------------------------------------------------------

# Function to solve N-Queen problem using iterative backtracking
def solve_n_queens_iterative(n):

    # positions[row] = column position of queen in that row
    positions = [-1] * n

    # Start from first row
    row = 0

    # List to store all solutions
    solutions = []

    # Continue until we backtrack before the first row
    while row >= 0:

        # Try next column in the current row
        positions[row] += 1

        # Find a safe position in the current row
        while positions[row] < n:

            safe = True

            # Check current queen against all previously placed queens
            for previous_row in range(row):

                # Check if queens are in the same column
                if positions[previous_row] == positions[row]:
                    safe = False
                    break

                # Check if queens are on the same diagonal
                if abs(positions[previous_row] - positions[row]) == \
                   abs(previous_row - row):

                    safe = False
                    break

            # If safe position found, stop checking columns
            if safe:
                break

            # Otherwise try next column
            positions[row] += 1

        # If a valid column is found
        if positions[row] < n:

            # If this is the last row,
            # one complete solution is found
            if row == n - 1:

                solutions.append(positions.copy())

            else:

                # Move to next row
                row += 1

                # Reset next row
                positions[row] = -1

        else:

            # No valid position in this row
            # Backtrack to previous row
            positions[row] = -1
            row -= 1

    return solutions


# Function to display chess board
def print_board(solution):

    n = len(solution)

    for row in range(n):

        for column in range(n):

            if solution[row] == column:
                print("Q", end=" ")

            else:
                print(".", end=" ")

        print()

    print()


# ---------------------------------------------------------
# MAIN PROGRAM
# ---------------------------------------------------------

# Take input from user
# If blank input is given, default value 4 is used

user_input = input(
    "Enter the number of queens [Press Enter for 4]: "
).strip()

# Check whether input is empty
if user_input == "":
    n = 4

else:
    try:
        n = int(user_input)

    except ValueError:
        print("Invalid input. Using default value N = 4.")
        n = 4


# Check that N is valid
if n <= 0:

    print("Number of queens must be greater than zero.")

else:

    print("\nNumber of Queens =", n)

    # Find solutions
    solutions = solve_n_queens_iterative(n)

    # Display result
    print("\nTotal number of solutions =", len(solutions))

    # If no solution exists
    if len(solutions) == 0:

        print("\nNo solution exists for N =", n)

    else:

        # Display every solution
        for i, solution in enumerate(solutions, start=1):

            print("\nSolution", i)

            print("Queen positions:", solution)

            print_board(solution)


Number of Queens = 4

Total number of solutions = 2

Solution 1
Queen positions: [1, 3, 0, 2]
. Q . . 
. . . Q 
Q . . . 
. . Q . 


Solution 2
Queen positions: [2, 0, 3, 1]
. . Q . 
Q . . . 
. . . Q 
. Q . . 



In [ ]:
# Q2 Water Jug Problem using Breadth First Search
from collections import deque
def water_jug(capacity_a, capacity_b, target):

    start = (0, 0)

    # Queue contains:
    # current state and path used to reach that state
    queue = deque([(start, [])])

    visited = {start}

    while queue:

        (a, b), path = queue.popleft()

        # Goal test
        if a == target or b == target:
            return path, (a, b)

        # Calculate possible pouring quantities
        pour_a_to_b = min(a, capacity_b - b)
        pour_b_to_a = min(b, capacity_a - a)

        next_states = [

            ("Fill Jug A",
             (capacity_a, b)),

            ("Fill Jug B",
             (a, capacity_b)),

            ("Empty Jug A",
             (0, b)),

            ("Empty Jug B",
             (a, 0)),

            ("Pour A -> B",
             (a - pour_a_to_b,
              b + pour_a_to_b)),

            ("Pour B -> A",
             (a + pour_b_to_a,
              b - pour_b_to_a))]

        for action, state in next_states:

            if state not in visited:

                visited.add(state)

                queue.append(
                    (state, path + [(action, state)]))

    return None, None


# Main Program

capacity_a = 4
capacity_b = 3
target = 2

path, final_state = water_jug(
    capacity_a,
    capacity_b,
    target
)

print("Initial State: (0, 0)")

if path:

    for action, state in path:
        print(action, "=>", state)

    print("\nGoal reached:", final_state)

else:
    print("No solution exists.")

Initial State: (0, 0)
Fill Jug B => (0, 3)
Pour B -> A => (3, 0)
Fill Jug B => (3, 3)
Pour B -> A => (4, 2)

Goal reached: (4, 2)


In [ ]:
# Q3 Minimax Algorithm
def minimax(depth, node_index,
            maximizing_player,
            scores, maximum_depth):

    # Base case:
    # Leaf node has been reached
    if depth == maximum_depth:
        return scores[node_index]

    # MAX player's turn
    if maximizing_player:

        left_value = minimax(
            depth + 1,
            node_index * 2,
            False,
            scores,
            maximum_depth)

        right_value = minimax(
            depth + 1,
            node_index * 2 + 1,
            False,
            scores,
            maximum_depth)

        return max(left_value, right_value)

    # MIN player's turn
    else:

        left_value = minimax(
            depth + 1,
            node_index * 2,
            True,
            scores,
            maximum_depth)

        right_value = minimax(
            depth + 1,
            node_index * 2 + 1,
            True,
            scores,
            maximum_depth)

        return min(left_value, right_value)


# Terminal values of the game tree
scores = [3, 5, 2, 9, 12, 5, 23, 23]

maximum_depth = 3

optimal_value = minimax(
    0,
    0,
    True,
    scores,
    maximum_depth)

print("Leaf node values:", scores)
print("Optimal value =", optimal_value)

Leaf node values: [3, 5, 2, 9, 12, 5, 23, 23]
Optimal value = 12


In [ ]:
#  Q4 - AO* Algorithm on an AND-OR graph
class AOStar:

    def __init__(self, graph, heuristic):

        self.graph = graph
        self.h = heuristic.copy()

        # Nodes for which solution has been determined
        self.solved = set()

        # Stores best selected children
        self.solution = {}


    def solve(self, node):

        # Terminal node
        if not self.graph.get(node):

            self.h[node] = 0
            self.solved.add(node)

            return 0


        while node not in self.solved:

            alternatives = []

            # Calculate estimated cost
            # for every alternative
            for children, edge_cost in self.graph[node]:

                estimated_cost = (
                    edge_cost
                    + sum(self.h[child]
                          for child in children))

                alternatives.append(
                    (estimated_cost,
                     children,
                     edge_cost))


            # Select minimum-cost alternative
            _, best_children, best_edge_cost = min(
                alternatives,
                key=lambda x: x[0])

            self.solution[node] = best_children


            # Solve selected children
            for child in best_children:

                if child not in self.solved:
                    self.solve(child)


            # Recalculate cost after child values
            # have been updated
            alternatives = []

            for children, edge_cost in self.graph[node]:

                estimated_cost = (
                    edge_cost
                    + sum(self.h[child]
                          for child in children))

                alternatives.append(
                    (estimated_cost,
                     children,
                     edge_cost))


            best_cost, best_children, _ = min(
                alternatives,
                key=lambda x: x[0])

            self.h[node] = best_cost

            self.solution[node] = best_children


            # A node is solved when all nodes
            # belonging to its selected alternative
            # are solved
            if all(
                child in self.solved
                for child in best_children):

                self.solved.add(node)


        return self.h[node]


    def print_solution(self, node, indent=0):

        print(" " * indent + node, end="")

        if node in self.solution:

            children = self.solution[node]

            if len(children) > 1:

                print(
                    " -> ("
                    + " AND ".join(children)
                    + ")")

            else:

                print(" -> " + children[0])


            for child in children:

                self.print_solution(
                    child,
                    indent + 4)

        else:

            print()


# AND-OR Graph
# Each entry:
# Node : [(children, edge cost)]

graph = {

    "A": [
        (["B", "C"], 1),
        (["D"], 2)],

    "B": [
        (["E"], 1),
        (["F"], 2)],

    "C": [
        (["G"], 2)],

    "D": [],
    "E": [],
    "F": [],
    "G": []}


# Initial heuristic values
heuristic = {

    "A": 10,
    "B": 4,
    "C": 2,
    "D": 6,
    "E": 3,
    "F": 2,
    "G": 0}

# Create AO* object
ao = AOStar(graph, heuristic)

optimal_cost = ao.solve("A")

print("Optimal Cost =", optimal_cost)

print("\nAO* Solution Graph:")

ao.print_solution("A")

Optimal Cost = 4

AO* Solution Graph:
A -> (B AND C)
    B -> E
        E
    C -> G
        G


In [ ]:
#  Q5 Gaussian Naive Bayes Classification
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report)

# Load Iris dataset
iris = load_iris()

X = iris.data
y = iris.target


print("Feature names:")
print(iris.feature_names)

print("\nClass names:")
print(iris.target_names)


# Split dataset
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y)


# Create Gaussian Naive Bayes model
model = GaussianNB()


# Train the model
model.fit(X_train, y_train)


# Predict test data
y_pred = model.predict(X_test)


# Evaluate the model
accuracy = accuracy_score(
    y_test,
    y_pred)


print("\nAccuracy:",
      round(accuracy * 100, 2),
      "%")


print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        y_pred))


print("\nClassification Report:")

print(
    classification_report(

        y_test,

        y_pred,

        target_names=iris.target_names))


# Example prediction
new_flower = [[5.1, 3.5, 1.4, 0.2]]

prediction = model.predict(new_flower)

print(
    "Predicted flower:",
    iris.target_names[prediction[0]])

Feature names:
['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']

Class names:
['setosa' 'versicolor' 'virginica']

Accuracy: 96.67 %

Confusion Matrix:
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30

Predicted flower: setosa


In [ ]:
# Q6 Logistic Regression Classification
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report)


# Load dataset
data = load_breast_cancer()

X = data.data
y = data.target


print("Target classes:")
print(data.target_names)


# Split dataset
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y)


# Create a pipeline:
# Standardization + Logistic Regression
model = make_pipeline(

    StandardScaler(),

    LogisticRegression(
        max_iter=1000,
        random_state=42))


# Train model
model.fit(
    X_train,
    y_train)


# Predict test values
y_pred = model.predict(
    X_test)


# Calculate accuracy
accuracy = accuracy_score(
    y_test,
    y_pred)


print(
    "Accuracy:",
    round(accuracy * 100, 2),
    "%")


print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        y_pred))


print("\nClassification Report:")

print(
    classification_report(

        y_test,
        y_pred,

        target_names=data.target_names))

Target classes:
['malignant' 'benign']
Accuracy: 98.25 %

Confusion Matrix:
[[41  1]
 [ 1 71]]

Classification Report:
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        42
      benign       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114



In [ ]:
# Q7
# ID3 Decision Tree from scratch
# Real-world example: Bank Loan Approval
import pandas as pd
from math import log2
from pprint import pprint


# -------------------------------------------------
# Step 1: Create Loan Approval Dataset
# -------------------------------------------------

data = pd.DataFrame({

    "Income": [
        "High", "High", "Medium", "Low",
        "Low", "Low", "Medium", "High",
        "High", "Low", "Medium", "Medium",
        "High", "Low"],

    "Credit": [
        "Good", "Good", "Good", "Good",
        "Poor", "Poor", "Poor", "Good",
        "Poor", "Good", "Poor", "Good",
        "Poor", "Good"],

    "Employed": [
        "Yes", "No", "Yes", "Yes",
        "Yes", "No", "No", "Yes",
        "Yes", "No", "Yes", "No",
        "No", "Yes"],

    "Debt": [
        "Low", "High", "Low", "Low",
        "High", "High", "Low", "High",
        "Low", "Low", "Low", "High",
        "High", "High"],

    "Approved": [
        "Yes", "No", "Yes", "Yes",
        "No", "No", "No", "Yes",
        "Yes", "No", "Yes", "No",
        "No", "Yes"]})


# -------------------------------------------------
# Step 2: Calculate entropy
# -------------------------------------------------

def entropy(column):

    probabilities = column.value_counts(
        normalize=True)

    result = 0

    for p in probabilities:

        result -= p * log2(p)

    return result


# -------------------------------------------------
# Step 3: Calculate Information Gain
# -------------------------------------------------

def information_gain(
        dataframe,
        feature,
        target="Approved"):

    total_entropy = entropy(
        dataframe[target])

    weighted_entropy = 0

    # Divide dataset based on feature values
    for value, subset in dataframe.groupby(feature):

        probability = (
            len(subset) /
            len(dataframe))

        weighted_entropy += (
            probability
            * entropy(subset[target]))

    gain = (
        total_entropy
        - weighted_entropy)

    return gain


# -------------------------------------------------
# Step 4: Build ID3 Tree
# -------------------------------------------------

def id3(
        dataframe,
        features,
        target="Approved",
        default=None):

    # If dataset becomes empty
    if dataframe.empty:
        return default


    labels = dataframe[target]


    # If all rows belong to same class
    if len(labels.unique()) == 1:
        return labels.iloc[0]


    # Majority class
    majority_class = labels.mode()[0]


    # If there are no features left
    if not features:
        return majority_class


    # Calculate information gain of each feature
    gains = {

        feature:
        information_gain(
            dataframe,
            feature,
            target)

        for feature in features}


    # Select feature having maximum gain
    best_feature = max(
        gains,
        key=gains.get)


    tree = {
        best_feature: {}}


    # Create branches
    for value in dataframe[best_feature].unique():

        subset = dataframe[
            dataframe[best_feature] == value
        ].drop(
            columns=[best_feature])


        remaining_features = [

            feature

            for feature in features

            if feature != best_feature]


        subtree = id3(

            subset,

            remaining_features,

            target,

            majority_class)


        tree[best_feature][value] = subtree


    return tree


# -------------------------------------------------
# Step 5: Prediction function
# -------------------------------------------------

def predict(tree, sample):

    if not isinstance(tree, dict):
        return tree


    root = next(iter(tree))

    value = sample[root]

    subtree = tree[root].get(value)


    if subtree is None:
        return "Unknown"


    return predict(
        subtree,
        sample)


# -------------------------------------------------
# Main Program
# -------------------------------------------------

features = [

    "Income",
    "Credit",
    "Employed",
    "Debt"]


print("Loan Approval Dataset:")

print(data)


print(
    "\nInitial Entropy:",
    entropy(data["Approved"]))


print("\nInformation Gain:")

for feature in features:

    print(
        feature,
        "=",
        round(
            information_gain(
                data,
                feature),4))


tree = id3(
    data,
    features)


print("\nGenerated ID3 Decision Tree:")

pprint(tree)


# Test a new applicant
new_applicant = {

    "Income": "Low",

    "Credit": "Good",

    "Employed": "Yes",

    "Debt": "High"}


prediction = predict(
    tree,
    new_applicant)


print("\nNew Applicant:")

print(new_applicant)


print(
    "Loan Approval Prediction:",
    prediction)

Loan Approval Dataset:
    Income Credit Employed  Debt Approved
0     High   Good      Yes   Low      Yes
1     High   Good       No  High       No
2   Medium   Good      Yes   Low      Yes
3      Low   Good      Yes   Low      Yes
4      Low   Poor      Yes  High       No
5      Low   Poor       No  High       No
6   Medium   Poor       No   Low       No
7     High   Good      Yes  High      Yes
8     High   Poor      Yes   Low      Yes
9      Low   Good       No   Low       No
10  Medium   Poor      Yes   Low      Yes
11  Medium   Good       No  High       No
12    High   Poor       No  High       No
13     Low   Good      Yes  High      Yes

Initial Entropy: 1.0

Information Gain:
Income = 0.0207
Credit = 0.0611
Employed = 0.6894
Debt = 0.1369

Generated ID3 Decision Tree:
{'Employed': {'No': 'No',
              'Yes': {'Income': {'High': 'Yes',
                                 'Low': {'Credit': {'Good': 'Yes',
                                                    'Poor': 'No'}},
   

In [ ]:
# Q8 Support Vector Machine Classification
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report)


# Load Iris dataset
iris = load_iris()

X = iris.data
y = iris.target


print("Class names:")
print(iris.target_names)


# Split dataset into 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,

    test_size=0.20,

    random_state=42,

    stratify=y)


# Create SVM pipeline
model = make_pipeline(

    StandardScaler(),

    SVC(
        kernel="rbf",
        C=1.0,
        gamma="scale"))


# Train SVM
model.fit(
    X_train,
    y_train)


# Predict test data
y_pred = model.predict(
    X_test)


# Accuracy
accuracy = accuracy_score(
    y_test,
    y_pred)


print(
    "\nAccuracy:",
    round(
        accuracy * 100,
        2),"%")


# Confusion matrix
print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        y_pred))


# Classification report
print("\nClassification Report:")

print(
    classification_report(

        y_test,

        y_pred,

        target_names=iris.target_names))


# Example new flower
new_flower = [[

    5.1,
    3.5,
    1.4,
    0.2]]


prediction = model.predict(
    new_flower)


print(
    "\nPredicted class:",
    iris.target_names[
        prediction[0]])

Class names:
['setosa' 'versicolor' 'virginica']

Accuracy: 96.67 %

Confusion Matrix:
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]

Classification Report:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30


Predicted class: setosa
